# Notebook 04: Ablation Studies

**Runs on:** Google Colab (T4 GPU)

**Runtime:** ~45 minutes on T4

Systematic experiments to understand what contributes to performance:
1. **SE Attention**: ResNet with vs without SE block
2. **Augmentation**: With vs without data augmentation
3. **Transfer Learning**: Pretrained vs from scratch
4. **Data Efficiency**: 10%, 25%, 50%, 100% training data

---
## Instructions
1. Upload to Colab, set T4 GPU
2. Run all cells
3. Download `ablation_results.zip`

In [ ]:
!pip install -q torch torchvision timm scikit-learn tqdm

In [ ]:
import os, time, copy, json, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import EuroSAT
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from collections import Counter
import timm

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
torch.backends.cudnn.deterministic = True
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

os.makedirs('results/models', exist_ok=True)
os.makedirs('results/figures', exist_ok=True)
os.makedirs('results/metrics', exist_ok=True)

In [ ]:
# Dataset setup (same as notebook 02)
raw_dataset = EuroSAT(root='./data', download=True)
CLASS_NAMES = ['AnnualCrop','Forest','HerbaceousVegetation','Highway','Industrial',
               'Pasture','PermanentCrop','Residential','River','SeaLake']
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

all_indices = list(range(len(raw_dataset)))
all_labels = [raw_dataset[i][1] for i in all_indices]
train_idx, temp_idx, train_labels, temp_labels = train_test_split(
    all_indices, all_labels, test_size=0.30, random_state=SEED, stratify=all_labels)
val_idx, test_idx, _, _ = train_test_split(
    temp_idx, temp_labels, test_size=0.50, random_state=SEED, stratify=temp_labels)

train_label_counts = Counter(train_labels)
class_weights = torch.tensor(
    [len(train_labels) / (len(CLASS_NAMES) * train_label_counts[i]) for i in range(len(CLASS_NAMES))],
    dtype=torch.float32).to(device)

class EuroSATSubset(torch.utils.data.Dataset):
    def __init__(self, dataset, indices, transform=None):
        self.dataset, self.indices, self.transform = dataset, indices, transform
    def __len__(self): return len(self.indices)
    def __getitem__(self, idx):
        img, label = self.dataset[self.indices[idx]]
        if self.transform: img = self.transform(img)
        return img, label

train_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.RandomHorizontalFlip(0.5), transforms.RandomVerticalFlip(0.5),
    transforms.RandomApply([transforms.RandomRotation(degrees=[90, 90])], p=0.5),
    transforms.RandomApply([transforms.ColorJitter(0.2, 0.2, 0.1)], p=0.3),
    transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
no_aug_transform = transforms.Compose([
    transforms.Resize((64, 64)), transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_transform = no_aug_transform

BATCH_SIZE = 64
train_dataset = EuroSATSubset(raw_dataset, train_idx, train_transform)
val_dataset = EuroSATSubset(raw_dataset, val_idx, eval_transform)
test_dataset = EuroSATSubset(raw_dataset, test_idx, eval_transform)
train_loader = DataLoader(train_dataset, BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
print(f"Train: {len(train_idx)} | Val: {len(val_idx)} | Test: {len(test_idx)}")

In [ ]:
# Model definitions (same as notebook 02)
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.squeeze = nn.AdaptiveAvgPool2d(1)
        self.excitation = nn.Sequential(
            nn.Linear(channels, channels//reduction, bias=False), nn.ReLU(True),
            nn.Linear(channels//reduction, channels, bias=False), nn.Sigmoid())
    def forward(self, x):
        b, c = x.size()[:2]
        y = self.excitation(self.squeeze(x).view(b, c)).view(b, c, 1, 1)
        return x * y.expand_as(x)

class ResNetSE(nn.Module):
    def __init__(self, num_classes=10, pretrained=True, use_se=True):
        super().__init__()
        self.use_se = use_se
        self.backbone = timm.create_model('resnet50', pretrained=pretrained, num_classes=0)
        if use_se: self.se_block = SEBlock(2048, 16)
        self.classifier = nn.Sequential(
            nn.Dropout(0.3), nn.Linear(2048, 512), nn.ReLU(True),
            nn.Dropout(0.2), nn.Linear(512, num_classes))
    def forward(self, x):
        f = self.backbone.forward_features(x)
        if self.use_se: f = self.se_block(f)
        return self.classifier(F.adaptive_avg_pool2d(f, 1).flatten(1))

In [ ]:
# Training functions (same as notebook 02)
class EarlyStopping:
    def __init__(self, patience=10):
        self.patience, self.counter, self.best_loss, self.should_stop = patience, 0, None, False
    def __call__(self, val_loss):
        if self.best_loss is None: self.best_loss = val_loss
        elif val_loss > self.best_loss - 1e-4:
            self.counter += 1
            if self.counter >= self.patience: self.should_stop = True
        else: self.best_loss, self.counter = val_loss, 0

def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    loss_sum, correct, total = 0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        out = model(imgs)
        loss = criterion(out, labels)
        loss.backward(); optimizer.step()
        loss_sum += loss.item()*imgs.size(0)
        correct += out.max(1)[1].eq(labels).sum().item()
        total += labels.size(0)
    return loss_sum/total, correct/total

@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    loss_sum, correct, total = 0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        out = model(imgs)
        loss_sum += criterion(out, labels).item()*imgs.size(0)
        correct += out.max(1)[1].eq(labels).sum().item()
        total += labels.size(0)
    return loss_sum/total, correct/total

@torch.no_grad()
def test_accuracy(model, loader):
    model.eval()
    preds, labels = [], []
    for imgs, lbl in loader:
        out = model(imgs.to(device))
        preds.extend(out.max(1)[1].cpu().numpy())
        labels.extend(lbl.numpy())
    return accuracy_score(labels, preds), f1_score(labels, preds, average='macro')

def train_ablation(model, t_loader, v_loader, name, epochs=30, lr=1e-4, patience=10):
    print(f"\n--- {name} ---")
    criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
    es = EarlyStopping(patience)
    best_acc, best_state = 0, None
    t0 = time.time()
    for ep in range(epochs):
        train_one_epoch(model, t_loader, criterion, optimizer)
        _, val_acc = evaluate(model, v_loader, criterion)
        scheduler.step()
        if val_acc > best_acc: best_acc, best_state = val_acc, copy.deepcopy(model.state_dict())
        es(1 - val_acc)  # using 1-acc as proxy for loss
        if es.should_stop: break
    model.load_state_dict(best_state)
    torch.save(model.state_dict(), f'results/models/{name}.pth')
    acc, f1 = test_accuracy(model, test_loader)
    print(f"  Done in {time.time()-t0:.0f}s | Test Acc: {acc:.4f} | F1: {f1:.4f}")
    return acc, f1

## Ablation 1: SE Attention Impact

In [ ]:
ablation = {}

# With SE (load from training notebook if available, else retrain)
model = ResNetSE(pretrained=True, use_se=True).to(device)
acc, f1 = train_ablation(model, train_loader, val_loader, 'Ablation_WithSE')
ablation['ResNet + SE'] = {'accuracy': acc, 'f1': f1}
del model; torch.cuda.empty_cache()

# Without SE
model = ResNetSE(pretrained=True, use_se=False).to(device)
acc, f1 = train_ablation(model, train_loader, val_loader, 'Ablation_NoSE')
ablation['ResNet (no SE)'] = {'accuracy': acc, 'f1': f1}
del model; torch.cuda.empty_cache()

print(f"\nSE Impact: {(ablation['ResNet + SE']['accuracy'] - ablation['ResNet (no SE)']['accuracy'])*100:+.2f}%")

## Ablation 2: Augmentation Impact

In [ ]:
# Without augmentation
train_noaug = EuroSATSubset(raw_dataset, train_idx, no_aug_transform)
loader_noaug = DataLoader(train_noaug, BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)

model = ResNetSE(pretrained=True, use_se=True).to(device)
acc, f1 = train_ablation(model, loader_noaug, val_loader, 'Ablation_NoAug')
ablation['ResNet+SE (no aug)'] = {'accuracy': acc, 'f1': f1}
ablation['ResNet+SE (with aug)'] = ablation['ResNet + SE']  # from ablation 1
del model; torch.cuda.empty_cache()

print(f"\nAug Impact: {(ablation['ResNet+SE (with aug)']['accuracy'] - ablation['ResNet+SE (no aug)']['accuracy'])*100:+.2f}%")

## Ablation 3: Transfer Learning Impact

In [ ]:
# From scratch (no pretrained weights)
model = ResNetSE(pretrained=False, use_se=True).to(device)
acc, f1 = train_ablation(model, train_loader, val_loader, 'Ablation_Scratch',
                          epochs=50, lr=1e-3, patience=15)
ablation['ResNet+SE (scratch)'] = {'accuracy': acc, 'f1': f1}
ablation['ResNet+SE (pretrained)'] = ablation['ResNet + SE']
del model; torch.cuda.empty_cache()

print(f"\nTransfer Learning Impact: {(ablation['ResNet+SE (pretrained)']['accuracy'] - ablation['ResNet+SE (scratch)']['accuracy'])*100:+.2f}%")

## Ablation 4: Data Efficiency

In [ ]:
data_efficiency = {}

for frac in [0.10, 0.25, 0.50, 1.0]:
    if frac < 1.0:
        n = int(len(train_idx) * frac)
        subset = EuroSATSubset(raw_dataset, train_idx[:n], train_transform)
        loader = DataLoader(subset, BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
        model = ResNetSE(pretrained=True, use_se=True).to(device)
        acc, f1 = train_ablation(model, loader, val_loader, f'Ablation_{int(frac*100)}pct')
        data_efficiency[frac] = {'accuracy': acc, 'f1': f1, 'n_samples': n}
        del model; torch.cuda.empty_cache()
    else:
        data_efficiency[frac] = {**ablation['ResNet + SE'], 'n_samples': len(train_idx)}

print("\nData Efficiency Summary:")
for frac, r in data_efficiency.items():
    print(f"  {frac*100:5.0f}% ({r['n_samples']:>5} samples): Acc={r['accuracy']:.4f} F1={r['f1']:.4f}")

## Results Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Ablation bar chart
abl_names = [k for k in ablation if k not in ['ResNet+SE (with aug)', 'ResNet+SE (pretrained)']]
abl_accs = [ablation[k]['accuracy'] * 100 for k in abl_names]
bar_colors = ['#e74c3c' if 'no' in k.lower() or 'scratch' in k.lower() else '#2ecc71' for k in abl_names]

axes[0].barh(abl_names, abl_accs, color=bar_colors, edgecolor='white')
axes[0].set_xlabel('Test Accuracy (%)', fontsize=12)
axes[0].set_title('Ablation Study Results', fontsize=14, fontweight='bold')
for i, v in enumerate(abl_accs):
    axes[0].text(v + 0.3, i, f'{v:.1f}%', va='center', fontsize=10)
axes[0].set_xlim(0, max(abl_accs) + 5)

# Data efficiency
fracs = sorted(data_efficiency.keys())
accs = [data_efficiency[f]['accuracy'] * 100 for f in fracs]
axes[1].plot([f*100 for f in fracs], accs, 'o-', color='#3498db', linewidth=2, markersize=10,
             markerfacecolor='white', markeredgewidth=2)
axes[1].set_xlabel('Training Data (%)', fontsize=12)
axes[1].set_ylabel('Test Accuracy (%)', fontsize=12)
axes[1].set_title('Data Efficiency: ResNet-50+SE', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3)
for f, a in zip(fracs, accs):
    axes[1].annotate(f'{a:.1f}%', (f*100, a), textcoords="offset points", xytext=(0,12), ha='center')

plt.tight_layout()
plt.savefig('results/figures/ablation_studies.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Save results
pd.DataFrame([{'Experiment': k, 'Accuracy': f"{v['accuracy']:.4f}", 'F1_Macro': f"{v['f1']:.4f}"}
              for k, v in ablation.items()
]).to_csv('results/metrics/ablation_results.csv', index=False)

pd.DataFrame([{'Fraction': f'{f:.0%}', 'Samples': r['n_samples'],
               'Accuracy': f"{r['accuracy']:.4f}", 'F1': f"{r['f1']:.4f}"}
              for f, r in data_efficiency.items()
]).to_csv('results/metrics/data_efficiency.csv', index=False)

# Zip and download
import shutil
shutil.make_archive('ablation_results', 'zip', '.', 'results')
try:
    from google.colab import files
    files.download('ablation_results.zip')
except ImportError:
    print("Download ablation_results.zip manually.")

## Summary

Completed 4 ablation experiments:
1. SE Attention: with vs without
2. Augmentation: with vs without
3. Transfer Learning: pretrained vs scratch
4. Data Efficiency: 10/25/50/100% training data

**Next:** Run `05_Interpretability.ipynb` and `06_Robustness_Quantization.ipynb` locally.